# 工具（Tools）
工具赋予智能体执行行动的能力。智能体超越了简单的仅模型工具绑定，实现了：

1. 序列中的多个工具调用（由单个提示触发）
2. 适当的并行工具调用
3. 基于先前结果的动态工具选择
4. 工具重试逻辑和错误处理
5. 工具调用之间的状态持久化

## 1. 定义工具

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
import os

@tool
def search(query: str) -> str:
    """搜索信息。"""
    return f"结果：{query}"

@tool
def get_weather(location: str) -> str:
    """获取位置的天气信息。"""
    return f"{location} 的天气：晴朗，72°F"

model = init_chat_model(
    model="qwen-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)
agent = create_agent(model, tools=[search, get_weather])

## 2. 工具错误处理

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """使用自定义消息处理工具执行错误。"""
    try:
        return handler(request)
    except Exception as e:
        # 向模型返回自定义错误消息
        return ToolMessage(
            content=f"工具错误：请检查您的输入并重试。({str(e)})",
            tool_call_id=request.tool_call["id"],
        )


agent = create_agent(
    model="qwen-plus", tools=[search, get_weather], middleware=[handle_tool_errors]
)
# 失败案例
# [
#     ...
#     ToolMessage(
#         content="工具错误：请检查您的输入并重试。(division by zero)",
#         tool_call_id="..."
#     ),
#     ...
# ]